# 06 · Ray on Union — and when you don't need it

Union runs Ray natively: a task environment can declare a `RayJobConfig`, and the platform
provisions a **transient, right-sized Ray cluster** (via KubeRay) for each execution, then
tears it down. You get Ray's programming model with Flyte's caching, retries, versioning,
and observability around it.

But a large share of "we need Ray" workloads are *embarrassingly parallel* — and for those,
the Union-native pattern from [04](./04-reusable-containers.ipynb) (fan-out over reusable
containers) is simpler to operate and often cheaper. This notebook shows the full extent of
the integration, then the same workload built **three ways**, and a decision framework.

> **Platform prerequisite:** the Ray plugin must be enabled in your dataplane (KubeRay
> operator + `enabled_plugins` in Helm). Check with your platform team —
> [appendix A](./appendix/A-platform-config-map.md) → *Compute plugins*.

In [ ]:
import asyncio
from typing import List

import flyte

flyte.init_from_config()

## 1. The extent of the integration

`flyteplugins-ray` gives you, per task execution:

- **An ephemeral Ray cluster**: head + worker groups, sized in code, torn down after the job
  (`shutdown_after_job_finishes`, `ttl_seconds_after_finished`)
- **Autoscaling worker groups**: `enable_autoscaling` + `min_replicas`/`max_replicas`
- **Per-group resources**: `requests`/`limits` or a full `pod_template` per worker group —
  e.g. a GPU worker group and a CPU preprocessing group in one cluster
- **Ray `runtime_env`** passthrough (pip packages, env vars for Ray workers)
- **Attach to an existing long-lived Ray cluster** instead: `address="ray://host:10001"`
- Ray dashboard access + logs wired into the Union UI (log links are configured in the
  dataplane Helm values)

Everything else about the task is normal Flyte: inputs/outputs, caching, retries, `File`/`Dir`
on GCS. The task function body becomes your Ray **driver**.

In [ ]:
from flyteplugins.ray import HeadNodeConfig, RayJobConfig, WorkerNodeConfig

ray_image = (
    flyte.Image.from_debian_base(name="workshop-ray", python_version=(3, 12))
    .with_pip_packages("ray[default]==2.46.0", "flyteplugins-ray==2.5.7")
)

ray_config = RayJobConfig(
    head_node_config=HeadNodeConfig(
        requests=flyte.Resources(cpu="1", memory="2Gi"),
    ),
    worker_node_config=[
        WorkerNodeConfig(
            group_name="cpu-workers",
            replicas=2,
            min_replicas=1,
            max_replicas=4,                       # used when autoscaling is on
            requests=flyte.Resources(cpu="2", memory="4Gi"),
        ),
        # A second, GPU-equipped group is just another entry:
        # WorkerNodeConfig(
        #     group_name="gpu-workers", replicas=1,
        #     requests=flyte.Resources(cpu="4", memory="16Gi",
        #                              gpu=flyte.GPU(device="T4", quantity=1)),
        # ),
    ],
    enable_autoscaling=True,
    runtime_env={"pip": ["numpy"]},               # extra deps for Ray workers
    shutdown_after_job_finishes=True,
    ttl_seconds_after_finished=300,
)

ray_env = flyte.TaskEnvironment(
    name="ray_jobs",
    image=ray_image,
    plugin_config=ray_config,                     # <- this makes it a Ray task env
    resources=flyte.Resources(cpu="2", memory="4Gi"),   # the driver pod itself
)

In [ ]:
@ray_env.task
async def ray_batch_score(texts: List[str]) -> List[float]:
    """The task body is the Ray driver: it runs on the ephemeral cluster's head."""
    import ray

    @ray.remote
    def score(text: str) -> float:
        return float(len(text)) * 2.0

    futures = [score.remote(t) for t in texts]
    return ray.get(futures)


# To attach to a long-lived, externally-managed Ray cluster instead:
# attached = RayJobConfig(
#     worker_node_config=[WorkerNodeConfig(group_name="default", replicas=2)],
#     address="ray://my-ray-head.ray-system:10001",
# )

### At the Kubernetes level

Running the cell below makes the plugin create a `RayJob` custom resource; the KubeRay
operator spins up head/worker pods in your dataplane, runs the driver, and cleans up after
`ttl_seconds_after_finished`. `kubectl get rayjobs,pods -n <project>-<domain>` while it runs.

In [ ]:
run = await flyte.run.aio(ray_batch_score, texts=[f"doc-{i}" for i in range(100)])
print(run.url)
await run.wait.aio()
print(run.outputs()[:5])

## 2. The same workload, three ways

The workload: score 100 documents in parallel. It's deliberately embarrassingly parallel —
the case where the choice of tool is least obvious and most consequential.

**Way 1 — Ray** (above): one Flyte action; parallelism *inside* the ephemeral cluster.
Per-item work is invisible to Union (no per-item cache/retry/UI); you operate KubeRay.
Cluster spin-up adds ~minutes of latency per execution.

**Way 2 — `flyte.map` over reusable containers:** every item is a first-class Flyte action —
individually cached, retried, observable — running on warm pods with no per-item cold start:

In [ ]:
from datetime import timedelta

reuse_image = (
    flyte.Image.from_debian_base(name="workshop-reuse", python_version=(3, 12))
    .with_pip_packages("unionai-reuse>=0.1.15")
)

pool_env = flyte.TaskEnvironment(
    name="score_pool",
    image=reuse_image,
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    reusable=flyte.ReusePolicy(
        replicas=(2, 6),
        concurrency=10,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=10),
    ),
)

driver_env = flyte.TaskEnvironment(
    name="score_driver",
    image=reuse_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    depends_on=[pool_env],
)


@pool_env.task(cache="auto")
async def score_one(text: str) -> float:
    return float(len(text)) * 2.0


@driver_env.task
async def map_batch_score(texts: List[str]) -> List[float]:
    out: List[float] = []
    async for r in flyte.map.aio(score_one, texts, return_exceptions=True):
        if isinstance(r, Exception):
            raise r
        out.append(r)
    return out


run = await flyte.run.aio(map_batch_score, texts=[f"doc-{i}" for i in range(100)])
print(run.url)
await run.wait.aio()
print(run.outputs()[:5])

**Way 3 — plain `asyncio.gather` in one task:** no extra infrastructure at all. Right when
per-item work is I/O-bound (API calls) or the whole batch fits comfortably in one pod:

In [ ]:
@driver_env.task
async def inline_batch_score(texts: List[str]) -> List[float]:
    async def score(text: str) -> float:
        await asyncio.sleep(0)            # stand-in for an awaited API call
        return float(len(text)) * 2.0

    return list(await asyncio.gather(*[score(t) for t in texts]))

## 3. Decision framework

| Question | If yes → |
|---|---|
| Do workers need to **talk to each other**? (allreduce, parameter servers, Ray Train/Tune/RLlib) | **Ray** |
| Do you need Ray's **shared-memory object store** (large arrays shared zero-copy across workers)? | **Ray** |
| Are you migrating an **existing Ray codebase** as-is? | **Ray** (ephemeral clusters per run) |
| Is the work **embarrassingly parallel** items/batches? | **`flyte.map` + ReusePolicy** |
| Do you want **per-item caching, retries, and UI visibility**? | **`flyte.map` + ReusePolicy** |
| Is it a moderate, **I/O-bound** fan-out inside one pod's capacity? | **`asyncio.gather`** |
| Do you mainly want **warm workers / model-in-memory** between items? | **ReusePolicy** (that's exactly what it does) |

Rules of thumb for the gray zone:

- **Batch inference** rarely needs Ray. A model loaded once per warm pod (`@alru_cache`)
  serving `flyte.map` traffic replicates Ray Serve's batch story with less machinery — and
  each item is cached/retryable.
- **Distributed data preprocessing** at the "1M independent items" scale is the
  micro-batching pattern from [04](./04-reusable-containers.ipynb), not a Ray Dataset job.
- **Distributed training** (multi-node, gradient sync) is genuinely Ray's home turf (or the
  PyTorch-elastic plugin) — keep it there, and let Flyte version/cache/schedule the job.
- **Operational cost** is asymmetric: reusable containers are a `ReusePolicy(...)` line;
  Ray on self-managed means the KubeRay operator, its upgrades, and Ray-version
  compatibility are part of your platform surface.

A practical migration heuristic for existing Ray code: if the only Ray APIs in the file are
`ray.remote` + `ray.get` over independent inputs, it ports to `flyte.map` in an afternoon —
Way 1 → Way 2 above is exactly that port.

## Further reading

- Next: [07-apps-and-inference](./07-apps-and-inference.ipynb) — real-time serving (vs the batch patterns here)
- Ray plugin reference: Union docs → Integrations → Ray

> 💬 **Discuss:** inventory your current or planned Ray workloads — which actually use collectives, Train/Tune, or the shared object store, and which are `ray.remote` fan-outs over independent inputs that would port to `flyte.map`?